# Waste Classification — Training Notebook (Google Colab)

Run this notebook in **Google Colab with T4 GPU** (Runtime → Change runtime type → T4 GPU).

## No manual file uploads needed

All code is pulled automatically from GitHub. The dataset is downloaded from Kaggle.
You only need to open this notebook in Colab — everything else is handled by the cells.

**How to open this notebook in Colab:**
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. File → Open notebook → **GitHub** tab
3. Paste: `https://github.com/K01J10W19/Waste_Classification_App`
4. Select `ml-training/notebooks/exploration.ipynb`

When the notebook is updated (bug fixes, improvements), just re-open from GitHub — you always get the latest version.

---

## First run (~35 min total)

1. Run all cells top-to-bottom
2. **Cell 1** mounts Google Drive and clones the repo from GitHub (one-time, ~30 sec)
3. **Cell 3** prompts for your `kaggle.json` API token, then installs dependencies (no restart needed)
4. Dataset downloads from Kaggle (~5 min), preprocessing creates splits on Drive (~10–15 min)
5. Training on T4 GPU: Phase 1 ~15 min + Phase 2 ~20 min

## Subsequent runs (splits already on Drive)

- Cell 1 mounts Drive and `git pull`s latest code automatically
- Cells skip download + preprocessing (splits already exist)
- Training starts within 2 min

---

## Steps covered
**Phase 2 (already validated locally):** class distribution · augmentation preview  
**Phase 3 (real training on T4):** GPU check · train (15+20 epochs) · evaluate · export to backend

## 0. Environment setup — run this first, every session

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
GITHUB_REPO = 'https://github.com/K01J10W19/Waste_Classification_App.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    REPO_ROOT  = DRIVE_ROOT / 'Waste_Classification_App'

    if (REPO_ROOT / '.git').exists():
        # Already a git repo — pull latest code
        print("Pulling latest code from GitHub...")
        result = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull'],
            capture_output=True, text=True
        )
        print(result.stdout.strip() or result.stderr.strip())

    elif REPO_ROOT.exists():
        # Folder exists (from manual upload) but not a git repo yet — initialise it
        print("Folder exists but not a git repo. Initialising and syncing from GitHub...")
        subprocess.run(['git', '-C', str(REPO_ROOT), 'init'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'add', 'origin', GITHUB_REPO], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/master'], check=True)
        print("Repository synced from GitHub.")

    else:
        # Folder doesn't exist at all — clone fresh
        print("Cloning repository from GitHub to Drive (one-time, ~30 sec)...")
        subprocess.run(['git', 'clone', GITHUB_REPO, str(REPO_ROOT)], check=True)
        print("Clone complete.")

else:
    # Local: walk up from CWD until we find the repo root
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / 'ml-training' / 'src').exists():
        if REPO_ROOT == REPO_ROOT.parent:
            raise RuntimeError("Cannot find repo root. Run from inside the project.")
        REPO_ROOT = REPO_ROOT.parent

# Create directories that must exist before training.
(REPO_ROOT / 'ml-training' / 'outputs' / 'checkpoints').mkdir(parents=True, exist_ok=True)
(REPO_ROOT / 'backend' / 'models').mkdir(parents=True, exist_ok=True)

# Add src/ to sys.path so scripts can import each other
SRC_DIR = str(REPO_ROOT / 'ml-training' / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# SPLITS_DIR points to where splits will be saved after Kaggle download + preprocessing.
SPLITS_DIR  = REPO_ROOT / 'ml-training' / 'data' / 'splits'
OUTPUTS_DIR = REPO_ROOT / 'ml-training' / 'outputs'

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"Splits dir : {SPLITS_DIR}")
splits_ok = (SPLITS_DIR / 'train').exists() and any((SPLITS_DIR / 'train').iterdir())
print(f"Splits ready: {'YES' if splits_ok else 'NO — run cells 5-6 to download from Kaggle'}")

In [ ]:
import os, subprocess, sys

# ── Kaggle API token setup ──────────────────────────────────────────────
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

if (kaggle_dir / "kaggle.json").exists():
    print("kaggle.json already present.")
elif IN_COLAB:
    print("Upload your kaggle.json API token when prompted below.")
    print("  1. Go to kaggle.com > profile icon > Settings")
    print("  2. Scroll to API section > Create New Token")
    print("  Note: if your file is named 'kaggle (1).json' that is fine.")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        (kaggle_dir / "kaggle.json").write_bytes(uploaded[filename])
        os.chmod(str(kaggle_dir / "kaggle.json"), 0o600)
        print(f"kaggle.json saved (uploaded as: {filename})")
    else:
        print("WARNING: No file uploaded. Dataset download will fail if splits are missing.")
else:
    print("Local environment — ensure ~/.kaggle/kaggle.json exists.")

# ── Install dependencies ────────────────────────────────────────────────
if IN_COLAB:
    # Colab already ships TF + JAX + ml_dtypes as a compatible stack.
    # Reinstalling TF would downgrade ml_dtypes and break JAX.
    # Only install packages Colab doesn't include.
    extra = ["kaggle>=1.6", "shap>=0.45", "lime>=0.2"]
    print("\nColab env — installing extra packages (kaggle, shap, lime) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + extra)
else:
    req_path = REPO_ROOT / "ml-training" / "requirements.txt"
    print(f"\nInstalling dependencies from {req_path} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)])

print("Dependencies installed.")

import tensorflow as tf
import numpy as np
print(f"TensorFlow : {tf.__version__}")
print(f"numpy      : {np.__version__}")


In [ ]:
## 1. Data — check splits or download from Kaggle

In [ ]:
# Download dataset from Kaggle if splits are not already on Drive
train_dir = SPLITS_DIR / "train"

if train_dir.exists() and any(train_dir.iterdir()):
    total = sum(1 for _ in SPLITS_DIR.rglob("*") if _.is_file())
    print(f"Pre-split data found ({total:,} files). Skipping download.")
else:
    print("Splits not found — downloading from Kaggle (~5 min, cloud-to-cloud)...")
    RAW_DIR = REPO_ROOT / "ml-training" / "data" / "raw"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download \
        -d alistairking/recyclable-and-household-waste-classification \
        -p {str(RAW_DIR)} --unzip
    print("Download complete. Run the next cell to create the train/val/test splits.")

In [ ]:
# Create train/val/test splits (only runs after a fresh Kaggle download)
if (SPLITS_DIR / "train").exists() and any((SPLITS_DIR / "train").iterdir()):
    print("Splits already present — skipping preprocessing.")
else:
    print("Creating splits from raw data (~10-15 min)...")
    !python {REPO_ROOT}/ml-training/src/data_preprocessing.py
    print("Splits saved to Drive — no need to redo this next session.")

In [ ]:
import matplotlib.pyplot as plt

splits = ["train", "val", "test"]
counts = {split: {} for split in splits}
for split in splits:
    split_path = SPLITS_DIR / split
    if not split_path.exists():
        continue
    for cls_dir in sorted(split_path.iterdir()):
        if cls_dir.is_dir():
            counts[split][cls_dir.name] = sum(1 for _ in cls_dir.rglob("*") if _.is_file())

classes = sorted(counts["train"].keys())
x = range(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, split in enumerate(splits):
    vals = [counts[split].get(c, 0) for c in classes]
    ax.bar([xi + i * width for xi in x], vals, width, label=split)

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(classes, rotation=20, ha="right")
ax.set_ylabel("Image count")
ax.set_title("Class distribution across splits")
ax.legend()
plt.tight_layout()
out = OUTPUTS_DIR / "class_distribution.png"
plt.savefig(str(out), dpi=150)
plt.show()
print(f"Saved to {out}")

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from augmentation import build_augmentation_layer

sample_cls = sorted(counts["train"].keys())[0]
sample_dir = SPLITS_DIR / "train" / sample_cls
sample_path = next(sample_dir.glob("*"))

img = tf.keras.utils.load_img(sample_path, target_size=(256, 256))
img_arr = tf.keras.utils.img_to_array(img)
img_tensor = tf.expand_dims(img_arr, axis=0)

augment = build_augmentation_layer()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0][0].imshow(img_arr.astype("uint8"))
axes[0][0].set_title("Original")
axes[0][0].axis("off")

for i, ax in enumerate(axes.flatten()[1:]):
    aug = augment(img_tensor, training=True)[0].numpy().clip(0, 255).astype("uint8")
    ax.imshow(aug)
    ax.set_title(f"Aug {i+1}")
    ax.axis("off")

fig.suptitle(f"Augmentation preview — {sample_cls}", fontsize=13)
plt.tight_layout()
out = OUTPUTS_DIR / "augmentation_preview.png"
plt.savefig(str(out), dpi=150)
plt.show()
print(f"Saved to {out}")

---
## Phase 3 — CNN Model (ResNet-50 Transfer Learning)

Steps covered:
1. Train ResNet-50 (Phase 1: head-only, Phase 2: fine-tuning)
2. Evaluate on test set → confusion matrix, accuracy/loss curves, classification report
3. Export best model to `backend/models/`

In [ ]:
# Set working directory to repo root so all !python commands below resolve correctly
%cd {REPO_ROOT}
print(f"CWD set to: {REPO_ROOT}")

## 5. GPU check

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {gpus}")
if gpus:
    for g in gpus:
        print(f"  {g}")
else:
    print("  No GPU detected — training will run on CPU (slow).")
    print("  In Colab: Runtime → Change runtime type → T4 GPU")

## 6. Train — Phase 1 (head-only) + Phase 2 (fine-tuning)

`train.py` runs both phases sequentially.  
Checkpoints are saved to `ml-training/outputs/checkpoints/`.

In [ ]:
# Adjust epochs as needed; EarlyStopping will cut short if val_accuracy plateaus.
# On Colab T4: Phase 1 ~15 min, Phase 2 ~20 min for this dataset size.
!python ml-training/src/train.py \
    --epochs-phase1 15 \
    --epochs-phase2 20 \
    --batch-size 32

## 7. Evaluate — confusion matrix, accuracy/loss curves, classification report

In [ ]:
!python {REPO_ROOT}/ml-training/src/evaluate.py

In [ ]:
from IPython.display import Image as IPImage, display

print("=== Confusion Matrix (Normalised) ===")
display(IPImage(str(OUTPUTS_DIR / 'confusion_matrix.png')))

print("\n=== Training History ===")
display(IPImage(str(OUTPUTS_DIR / 'training_history.png')))

## 8. Export model to backend

In [ ]:
!python {REPO_ROOT}/ml-training/src/export_model.py

In [ ]:
import json

backend_dir = REPO_ROOT / 'backend' / 'models'
print(f"Files in {backend_dir}:")
for f in sorted(backend_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:6.1f} MB")

with open(backend_dir / 'class_names.json') as f:
    print(f"\nclass_names: {json.load(f)}")

## Training complete

All outputs are automatically saved to your Google Drive folder.

| Output | Location |
|--------|----------|
| Checkpoints | `ml-training/outputs/checkpoints/` |
| Confusion matrix | `ml-training/outputs/confusion_matrix.png` |
| Training curves | `ml-training/outputs/training_history.png` |
| Classification report | `ml-training/outputs/classification_report.txt` |
| Trained model | `backend/models/waste_classifier.keras` |
| Legacy model | `backend/models/waste_classifier.h5` |
| Class names | `backend/models/class_names.json` |

**Next step — Phase 4:** Share `classification_report.txt` and `evaluation_results.json` to proceed with building the FastAPI backend.